# 00 · Quickstart: meet the emulator

A close-to-vanilla **Vision Transformer** trained to step E3SMv3 (1°, AMIP) forward by 6 hours.

* **Input:** 34 prognostic fields (PS, TS, and T, U, V, total water on 8 levels; level 0 is the model top, 7 is near the surface) plus 5 forcings (SOLIN, PHIS, land/ocean/ice fractions), normalized.
* **Tokens:** 4×4-degree patches, so 45 × 90 = **4050 tokens**, each with a learned position embedding. The embedding is the only way the model knows where it is on the globe.
* **Body:** 8 pre-LayerNorm blocks with 6 heads each and a GELU MLP. There is no windowing and no spectral layer, so all horizontal mixing is attention.
* **Output:** the 6-h **tendency** of the 34 fields. Next state = current state + tendency.

Every head and MLP writes additively into the residual stream, which makes it a good subject for circuit hunting. Trained on 2000–2009; everything here uses **2010**, which it never saw.

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
try:
    import vit_interp
except ImportError:  # not installed: use this checkout, $VIT_INTERP_REPO, or the shared one
    for repo in (Path.cwd().parent, Path(os.environ.get("VIT_INTERP_REPO", "/home/ac.ngmahfouz/vit-interp"))):
        if (repo / "vit_interp").is_dir():
            sys.path.insert(0, str(repo)); break
import numpy as np, torch, matplotlib.pyplot as plt
from vit_interp import playground as pg
em, meta = pg.load()      # or pg.load("/path/to/checkpoint.pt"); $PLAYGROUND_CKPT also works
FAST = pg.FAST            # PLAYGROUND_FAST=1 -> small settings (CPU tests)

## One step, against the truth

In [ ]:
t0 = "2010-01-15T00"
x = pg.state(t0)                                  # (1, 39, 180, 360), physical units
nxt, tend = pg.step(em, x)                        # emulated next state and 6-h tendency
tr = pg.truth(t0, steps=1)[0, :, :34]             # E3SM states at t0 and t0 + 6 h
true_tend = (tr[1] - tr[0]).to(tend.device)
pg.plot_maps({"ViT: V_3 tendency [m/s / 6 h]": tend[0, pg.ch("V_3")],
              "E3SM: V_3 tendency": true_tend[pg.ch("V_3")],
              "ViT − E3SM": tend[0, pg.ch("V_3")] - true_tend[pg.ch("V_3")]}, shared=True, cols=3);

How much better than persistence (tendency = 0) is one step, field by field?

In [ ]:
w = torch.tensor(np.cos(np.deg2rad(meta["lat"])))[:, None].to(tend.device)
def rmse(a): return float(((a ** 2) * w).sum() / (w.sum() * a.shape[-1])) ** 0.5
rows = []
for name in ["PS", "TS", "T_7", "T_3", "U_3", "V_3", "U_6", "specific_total_water_7"]:
    c = pg.ch(name)
    rows.append((name, rmse(tend[0, c] - true_tend[c]), rmse(true_tend[c])))
import pandas as pd
pd.DataFrame(rows, columns=["field", "RMSE ViT", "RMSE persistence"]).assign(
    skill=lambda d: 1 - d["RMSE ViT"] / d["RMSE persistence"]).round(4)

## A short forecast: where do errors grow?

In [ ]:
steps = 4 if FAST else 20                         # 20 steps = 5 days
emu, tru = pg.rollout(em, t0, steps)
lead_h = np.arange(steps + 1) * 6
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, name in zip(axes, ["PS", "T_7", "U_3", "V_3"]):
    c = pg.ch(name)
    err = [rmse((emu[k, c] - tru[k, c]).to(w.device)) for k in range(steps + 1)]
    pers = [rmse((tru[0, c] - tru[k, c]).to(w.device)) for k in range(steps + 1)]
    ax.plot(lead_h, err, label="ViT"); ax.plot(lead_h, pers, "--", label="persistence")
    ax.set_title(name); ax.set_xlabel("lead [h]")
axes[0].legend();

In [ ]:
k = steps
pg.plot_maps({f"ViT U_3 at +{6*k} h": emu[k, pg.ch("U_3")], f"E3SM U_3 at +{6*k} h": tru[k, pg.ch("U_3")],
              "difference": emu[k, pg.ch("U_3")] - tru[k, pg.ch("U_3")]}, cols=3, symmetric=False, cmap="viridis");

## Is it Earth?

A full-year free run with climatology maps, zonal means, drift and skill against persistence and climatology lives in the evaluation folders next to the checkpoints (`runs/vit-p4-d384-l8/eval-*`). The cell below shows the latest summary.

In [ ]:
evals = sorted(pg.RUN.glob("eval-*/summary.md"), key=os.path.getmtime)
print(evals[-1].read_text() if evals else "no evaluation yet")

## Try this

1. Pick another date (`pg.state("2010-07-20T12")`). Is the step worse in the summer hemisphere, or over the tropics?
2. Which fields does the ViT beat persistence on by the most, and which by the least? Look at the `skill` column for moisture and TS over land.
3. Plot the error at +5 days for `specific_total_water_7`. Where does it grow first: storm tracks, the ITCZ, or the poles?
4. 4°×4° patches: do you see **patch-edge artifacts** in the tendency maps? (Zoom in with `plt.xlim`/`plt.ylim`.)